In [3]:
#@title ⚙️ Setup **SPPIDER-seq**
#@markdown > This cell installs dependencies, loads the ESM-2 pre-trained PLM model, and downloads the PPI site prediction models.
#@markdown >
#@markdown > *It may take up to **5 minutes** to complete.*

%%time

try:
  print("Installing libraries and importing dependencies...")

  # Core libraries
  !pip install transformers biopython torch statsmodels scipy --quiet

  import os
  import re
  import random
  from io import StringIO
  from itertools import product
  from datetime import datetime
  import time
  import urllib.request

  import numpy as np
  import torch
  import torch.nn as nn
  import torch.nn.functional as F
  import matplotlib.pyplot as plt

  from Bio import SeqIO
  from IPython.display import display, Markdown, HTML
  import ipywidgets as ipw

  from scipy.stats import norm
  from statsmodels.stats.multitest import fdrcorrection

  # Device
  device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
  print("Using device:", device, "\n")

  # ESM-2 backbone (same as used in training)
  from transformers import EsmTokenizer, EsmModel

  ESM_MODEL_NAME = "facebook/esm2_t33_650M_UR50D"

  print(f"Loading ESM-2 checkpoint: {ESM_MODEL_NAME}...")
  tokenizer = EsmTokenizer.from_pretrained(ESM_MODEL_NAME)
  esm_model = EsmModel.from_pretrained(ESM_MODEL_NAME).to(device)

  for p in esm_model.parameters():
      p.requires_grad = False
  esm_model.eval()

  print("\nESM-2 checkpoint loaded")
  print("Note: You may safely ignore the warning about uninitialized pooler weights above. This component is not used in the prediction workflow and does not affect results.\n")

  # Working folders
  os.makedirs("models", exist_ok=True)
  os.makedirs("outputs", exist_ok=True)

  # Paths to the trained cross-attention checkpoints.
  # Only update these two URLs when you have NEW checkpoints.
  PEPTIDE_MODEL_URL = "https://raw.githubusercontent.com/aporollo-lab/SPPIDER-seq/main/models/crossattn_pep_run07_best.pt"
  RECEPTOR_MODEL_URL = "https://raw.githubusercontent.com/aporollo-lab/SPPIDER-seq/main/models/crossattn_rec_run27_best.pt"
  # Stable local filenames INSIDE the notebook – do NOT change these.
  PEPTIDE_MODEL_PATH  = "models/peptide_model.pt"
  RECEPTOR_MODEL_PATH = "models/receptor_model.pt"
  # Make sure these files are present in the 'models' folder
  print("Downloading PPI prediction models...")
  _ = urllib.request.urlretrieve(PEPTIDE_MODEL_URL,  PEPTIDE_MODEL_PATH)
  _ = urllib.request.urlretrieve(RECEPTOR_MODEL_URL, RECEPTOR_MODEL_PATH)

  print("Setup is complete\n")

except Exception as e:
    print("\n⚠️ Setup failed!")
    print("Error message:")
    print(e)


# ==== Global state for Colab session ====
output_sessions = []
current_output_folder = None

# Option: show prediction plots inline for each pair
show_plots = False  # toggled from the input-form checkbox

# ==== Benchmarking (prediction compute time only) ====
pred_time_total_s = 0.0
pred_time_pairs = {}  # (query_id, partner_id) -> seconds
pred_num_pairs = 0

# ==== Hyperparameters (must match training) ====
EMBED_DIM   = esm_model.config.hidden_size  # 1280 for esm2_t33_650M_UR50D
NUM_HEADS   = 16
MAX_TOKENS  = 1024   # ESM tokens including BOS/EOS
STRIDE      = 512    # residue stride
PRED_CUTOFF = 0.50   # for plotting / optional site filtering

# ==== Embedding cache ====
# Cache ESM embeddings per *sequence string* (on CPU).
# This avoids recomputing embeddings for the same protein used
# across multiple pairs, without tying the cache to FASTA IDs
# like "query" or "partner".
embedding_cache = {}  # maps sequence string -> (chunk_dicts, full_length)

EMBEDDING_CACHE_MAX_SEQS = 256    # max number of distinct sequences to cache
EMBEDDING_CACHE_MAX_LEN  = 3000   # only cache if sequence length <= this

def get_or_compute_embeddings(sequence, allow_cache=True):
    """
    Compute chunked ESM embeddings for a sequence, with optional caching.

    Returns:
        chunk_dicts: list of {"start_idx", "end_idx", "embedding" (CPU tensor)}
        full_len:    int, full sequence length
    """
    seq = sequence.strip()
    if not seq:
        return [], 0

    # Use the *sequence itself* as cache key, not the FASTA ID.
    seq_key = seq.upper()

    if allow_cache and seq_key in embedding_cache:
        return embedding_cache[seq_key]

    # No cache hit -> compute embeddings
    chunk_dicts, L = embed_sequence_chunks(seq)

    # Decide whether to store in cache
    if allow_cache and L <= EMBEDDING_CACHE_MAX_LEN and len(embedding_cache) < EMBEDDING_CACHE_MAX_SEQS:
        embedding_cache[seq_key] = (chunk_dicts, L)

    return chunk_dicts, L

# ==== Utility functions ====

def update_status(message, busy=False):
    """
    Update a dedicated status box (if defined) with a message and
    an optional CSS spinner animation.
    """
    if "status_box" not in globals():
        # If the UI cell hasn't defined status_box yet, just print.
        print(message)
        return

    with status_box:
        status_box.clear_output(wait=True)
        if busy:
            spinner_html = f"""
            <div style="display:flex;align-items:center;gap:8px;">
              <div style="
                  border: 4px solid #f3f3f3;
                  border-top: 4px solid #555;
                  border-radius: 50%;
                  width: 16px;
                  height: 16px;
                  animation: spin 0.8s linear infinite;
              "></div>
              <span>{message}</span>
            </div>
            <style>
            @keyframes spin {{
              0%   {{ transform: rotate(0deg); }}
              100% {{ transform: rotate(360deg); }}
            }}
            </style>
            """
            display(HTML(spinner_html))
        else:
            display(HTML(f"<span>{message}</span>"))

def parse_fasta(text):
    """Return list of (index, id, seq) from FASTA text."""
    records = list(SeqIO.parse(StringIO(text.strip()), "fasta"))
    return [(i, rec.id, str(rec.seq)) for i, rec in enumerate(records)]

def sanitize_filename(text):
    return re.sub(r'[^a-zA-Z0-9_.-]', '_', text)

def scramble_sequence(seq, seed=None):
    rng = random.Random(seed)
    arr = list(seq)
    rng.shuffle(arr)
    return ''.join(arr)

def compute_per_residue_z_and_p(real_pred, scrambled_pred_stack):
    """
    real_pred: (L,)
    scrambled_pred_stack: (N_scrambles, L)
    """
    mean_scr = np.mean(scrambled_pred_stack, axis=0)
    std_scr  = np.std(scrambled_pred_stack, axis=0)
    std_scr[std_scr == 0] = 1e-6  # prevent divide-by-zero

    z_scores = (real_pred - mean_scr) / std_scr
    p_values = 1.0 - norm.cdf(z_scores)
    _, q_values = fdrcorrection(p_values, alpha=0.05)
    return z_scores, p_values, q_values

def save_predictions(filename, seq_id, partner_id, model_type, sequence, probabilities,
                     p_values=None, q_values=None):
    """
    Write a tab-delimited file with per-residue probabilities and (optionally) p/FDR.
    sequence and probabilities must have the same length.

    NOTE: uses global `use_null_background` defined in the Input cell.
    """
    probs = np.asarray(probabilities, dtype=float)
    assert len(sequence) == len(probs), "sequence / probability length mismatch"

    with open(filename, "w") as f:
        f.write(f"# Query:{seq_id} Partner:{partner_id} Model:{model_type}-centric\n")
        if use_null_background and p_values is not None and q_values is not None:
            f.write("Position\tAminoAcid\tProbability\tP-value\tFDR\n")
            for i, (aa, prob, p, q) in enumerate(zip(sequence, probs, p_values, q_values), 1):
                f.write(f"{i}\t{aa}\t{prob:.3f}\t{p:.4g}\t{q:.4g}\n")
        else:
            f.write("Position\tAminoAcid\tProbability\n")
            for i, (aa, prob) in enumerate(zip(sequence, probs), 1):
                f.write(f"{i}\t{aa}\t{prob:.3f}\n")

# ==== ESM chunked embedding (residue-wise windows) ====

def embed_sequence_chunks(sequence, max_tokens=MAX_TOKENS, stride=STRIDE):
    """
    Chunk a single amino-acid sequence, embed each chunk with ESM, and
    return a list of dicts:
      [{"start_idx": int, "end_idx": int, "embedding": [L_chunk, D]}, ...], L_full
    """
    esm_model.eval()
    device_local = next(esm_model.parameters()).device

    win = max_tokens - 2  # account for BOS/EOS
    L   = len(sequence)

    if L == 0:
        return [], 0

    chunk_data = []
    for start in range(0, L, stride):
        end = min(start + win, L)
        if end <= start:
            break

        enc = tokenizer(sequence[start:end],
                        return_tensors="pt",
                        add_special_tokens=True)
        input_ids = enc["input_ids"].to(device_local)

        with torch.no_grad():
            out = esm_model(input_ids)
            # [1, T_tokens, D] -> strip BOS/EOS -> [L_chunk, D]
            emb = out.last_hidden_state[:, 1:-1, :].squeeze(0).contiguous()

        chunk_data.append({
            "start_idx": start,
            "end_idx": end,
            "embedding": emb.cpu()
        })

        if end == L:
            break

    return chunk_data, L

# ==== Cross-attention model (must match training script) ====

class CrossAttentionLayer(nn.Module):
    def __init__(self, embed_dim, num_heads):
        super().__init__()
        self.cross_attn = nn.MultiheadAttention(embed_dim, num_heads, batch_first=True)
        self.norm = nn.LayerNorm(embed_dim)

    def forward(self, query, context, context_mask=None):
        # query: [B, Lq, D], context: [B, Lc, D]
        out, _ = self.cross_attn(query, context, context, key_padding_mask=context_mask)
        return self.norm(query + out)

class ChunkwiseInteractionModel(nn.Module):
    def __init__(self, embed_dim=EMBED_DIM, num_heads=NUM_HEADS, initial_bias=None):
        super().__init__()
        self.cross = CrossAttentionLayer(embed_dim, num_heads)
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, embed_dim),
            nn.ReLU(),
            nn.Dropout(0.10),
            nn.Linear(embed_dim, 1)
        )
        if initial_bias is not None:
            with torch.no_grad():
                self.mlp[-1].bias.fill_(initial_bias)

    def forward(self, q_chunks, c_chunks):
        """
        q_chunks, c_chunks: lists of [L, D] tensors (on device)
        Returns a list of [L_chunk] logit vectors, one per query chunk.
        """
        pooled_per_q = []
        for q in q_chunks:
            if q.ndim == 2:
                q = q.unsqueeze(0)  # [1, Lq, D]
            ctx_logits = []
            for c in c_chunks:
                if c.ndim == 2:
                    c = c.unsqueeze(0)  # [1, Lc, D]
                ctx_mask = (c.abs().sum(dim=-1) == 0)  # [1, Lc]
                x = self.cross(q, c, context_mask=ctx_mask)   # [1, Lq, D]
                logits = self.mlp(x).squeeze(0).squeeze(-1)   # [Lq]
                ctx_logits.append(logits)
            pooled = torch.max(torch.stack(ctx_logits, dim=0), dim=0).values  # [Lq]
            pooled_per_q.append(pooled)
        return pooled_per_q

# ======================  Logit merge  ======================

def merge_chunk_logits_to_full(pooled_per_q, q_starts, full_len, device=device):
    """
    pooled_per_q: list of [L_chunk] logit tensors
    q_starts: list of starting indices for each chunk
    """
    out = torch.full((full_len,), -1e9, dtype=torch.float32, device=device)
    for vec, st in zip(pooled_per_q, q_starts):
        vec = vec.to(device)
        Lq = vec.shape[0]
        placed = torch.full((full_len,), -1e9, dtype=torch.float32, device=device)
        placed[st:st+Lq] = vec
        out = torch.maximum(out, placed)
    out[out <= -1e8] = -30.0
    return out

# ==== Load trained cross-attention models ====

peptide_site_model = None
receptor_site_model = None

def _safe_torch_load(path, map_location=None):
    """
    Torch 2.6+ defaults to weights_only=True, which can break older checkpoints.
    This helper first tries the default behavior, and if that fails, retries with
    weights_only=False (trusted file).
    """
    try:
        return torch.load(path, map_location=map_location)
    except TypeError:
        # Older torch that doesn't know weights_only kwarg
        raise
    except Exception:
        # Retry with full unpickling (old behavior)
        return torch.load(path, map_location=map_location, weights_only=False)

def _find_existing_model_path(primary):
    """
    Helper: return the first existing path among [primary].
    Raises FileNotFoundError with a clear message if nothing is found.
    """
    candidates = [primary]
    for p in candidates:
        if p and os.path.exists(p):
            return p
    raise FileNotFoundError(
        f"Some of the expected model files were not found: {candidates}. "
        "Please check Cell 0: it should download/copy the models into 'models/'."
    )

def load_ppi_models():
    """
    Load peptide- and receptor-centric models using the generic paths
    defined in Cell 0: PEPTIDE_MODEL_PATH and RECEPTOR_MODEL_PATH.
    """
    global peptide_site_model, receptor_site_model

    pep_path = _find_existing_model_path(PEPTIDE_MODEL_PATH)
    rec_path = _find_existing_model_path(RECEPTOR_MODEL_PATH)

    if peptide_site_model is None:
        print(f"Loading peptide-centric model from {pep_path}")
        m = ChunkwiseInteractionModel(embed_dim=EMBED_DIM, num_heads=NUM_HEADS).to(device)
        state = _safe_torch_load(pep_path, map_location=device)
        m.load_state_dict(state, strict=True)
        m.eval()
        peptide_site_model = m

    if receptor_site_model is None:
        print(f"Loading receptor-centric model from {rec_path}")
        m = ChunkwiseInteractionModel(embed_dim=EMBED_DIM, num_heads=NUM_HEADS).to(device)
        state = _safe_torch_load(rec_path, map_location=device)
        m.load_state_dict(state, strict=True)
        m.eval()
        receptor_site_model = m

# ==== Single-pair prediction helpers ====

def predict_query_given_context(query_seq, context_seq, model,
                                cache_query=True, cache_context=True):
    """
    Predict per-residue probabilities ONLY for `query_seq`,
    using `context_seq` only as cross-attention context.

    Returns:
        probs: np.ndarray shape (len(query_seq),)
    """
    # Query embeddings (cached)
    q_chunk_dicts, q_len = get_or_compute_embeddings(query_seq, allow_cache=cache_query)

    # Context embeddings (optionally cached)
    if cache_context:
        c_chunk_dicts, _ = get_or_compute_embeddings(context_seq, allow_cache=True)
    else:
        c_chunk_dicts, _ = embed_sequence_chunks(context_seq)

    # Move to device
    q_chunks = [d["embedding"].to(device) for d in q_chunk_dicts]
    c_chunks = [d["embedding"].to(device) for d in c_chunk_dicts]
    q_starts = [int(d["start_idx"]) for d in q_chunk_dicts]

    if len(q_chunks) == 0 or len(c_chunks) == 0:
        return np.zeros(len(query_seq), dtype=float)

    with torch.no_grad():
        pooled_per_q = model(q_chunks, c_chunks)
        logits_full  = merge_chunk_logits_to_full(pooled_per_q, q_starts, q_len)
        probs        = torch.sigmoid(logits_full).cpu().numpy()

    # Ensure exact length = len(query_seq)
    if len(probs) > len(query_seq):
        probs = probs[:len(query_seq)]
    elif len(probs) < len(query_seq):
        probs = np.pad(probs, (0, len(query_seq) - len(probs)), "edge")

    return probs

def compute_scrambled_predictions_for_query(query_seq, partner_seq, model,
                                            num_scrambles=100, base_seed=0):
    """
    Null distribution by scrambling PARTNER while keeping QUERY fixed.

    - Query embeddings: cached/reused
    - Scrambled partner embeddings: NOT cached

    Returns: array shape (num_scrambles, len(query_seq))
    """
    all_scrambled = []
    for k in range(num_scrambles):
        seed = base_seed + k
        scrambled_partner = scramble_sequence(partner_seq, seed=seed)

        probs = predict_query_given_context(
            query_seq=query_seq,
            context_seq=scrambled_partner,
            model=model,
            cache_query=True,
            cache_context=False  # do NOT cache scrambled partners
        )
        all_scrambled.append(probs)

    return np.stack(all_scrambled, axis=0)

# ==== High-level workflow over all FASTA pairs ====

def predict_query_two_views(query_seq, partner_seq, cache_query=True, cache_partner=True):
    """
    Return two probability vectors over QUERY positions only:
      - query as receptor (receptor-centric model)
      - query as ligand/peptide (peptide-centric model)

    partner_seq is context only in both cases.
    """
    # Query predicted as receptor, partner is ligand context
    p_as_receptor = predict_query_given_context(
        query_seq=query_seq,
        context_seq=partner_seq,
        model=receptor_site_model,
        cache_query=cache_query,
        cache_context=cache_partner
    )

    # Query predicted as peptide/ligand, partner is receptor context
    p_as_peptide = predict_query_given_context(
        query_seq=query_seq,
        context_seq=partner_seq,
        model=peptide_site_model,
        cache_query=cache_query,
        cache_context=cache_partner
    )

    return p_as_receptor, p_as_peptide

def run_ppi_for_pair(query_id, query_seq, partner_id, partner_seq, output_dir, num_scrambles=5):
    """
    For one (query, partner) pair, produce TWO probability tracks over the QUERY positions only:
      1) query-as-receptor  (receptor model)
      2) query-as-peptide   (peptide model)

    partner is context only.
    """
    print(f"  {query_id} ↔ {partner_id}")

    global pred_time_total_s, pred_time_pairs, pred_num_pairs
    t0 = time.perf_counter()

    # Compute two "views" over query residues
    p_rec, p_pep = predict_query_two_views(query_seq, partner_seq)

    # Optional null-background significance (scramble partner only)
    rec_pvals = rec_qvals = None
    pep_pvals = pep_qvals = None

    if use_null_background:
        print("    Computing null background (query-as-receptor)...")
        scr_rec = compute_scrambled_predictions_for_query(
            query_seq=query_seq,
            partner_seq=partner_seq,
            model=receptor_site_model,
            num_scrambles=num_scrambles
        )
        _, rec_pvals, rec_qvals = compute_per_residue_z_and_p(p_rec, scr_rec)

        print("    Computing null background (query-as-peptide)...")
        scr_pep = compute_scrambled_predictions_for_query(
            query_seq=query_seq,
            partner_seq=partner_seq,
            model=peptide_site_model,
            num_scrambles=num_scrambles
        )
        _, pep_pvals, pep_qvals = compute_per_residue_z_and_p(p_pep, scr_pep)

    t1 = time.perf_counter()
    dt = t1 - t0

    pred_time_total_s += dt
    pred_num_pairs += 1
    pred_time_pairs[(query_id, partner_id)] = dt

    safe_q = sanitize_filename(query_id)
    safe_p = sanitize_filename(partner_id)

    # Save query-length probability files only
    rec_file = os.path.join(output_dir, f"{safe_q}__{safe_p}__query_as_receptor.txt")
    pep_file = os.path.join(output_dir, f"{safe_q}__{safe_p}__query_as_peptide.txt")

    save_predictions(rec_file, query_id, partner_id, "query_as_receptor", query_seq, p_rec, rec_pvals, rec_qvals)
    save_predictions(pep_file, query_id, partner_id, "query_as_peptide",  query_seq, p_pep, pep_pvals, pep_qvals)

    # Combined plot over QUERY positions
    fig_name = os.path.join(output_dir, f"{safe_q}__{safe_p}__query_combined_plot.png")
    L = len(query_seq)
    x = np.arange(1, L + 1)

    fig, ax = plt.subplots(figsize=(10, 4), dpi=300)

    # Curves
    ax.plot(x, p_rec, label="Receptor-centric model", linewidth=2, color='orange')
    ax.plot(x, p_pep, label="Peptide-centric model", linestyle="--", linewidth=2, color='blue')

    # Optional significance overlay as red dots
    if use_null_background and (rec_qvals is not None) and (pep_qvals is not None):
        rec_q = np.asarray(rec_qvals, dtype=float)
        pep_q = np.asarray(pep_qvals, dtype=float)

        # Safety: ensure lengths match query
        if rec_q.shape[0] != L or pep_q.shape[0] != L:
            raise ValueError(
                f"Length mismatch: L={L}, rec_q={rec_q.shape[0]}, pep_q={pep_q.shape[0]}"
            )

        rec_sig = rec_q < 0.05
        pep_sig = pep_q < 0.05

        # Red dots placed directly on the corresponding curve values
        if np.any(rec_sig):
            ax.scatter(x[rec_sig], np.asarray(p_rec)[rec_sig],
                      s=14, color="red", edgecolors="none",
                      zorder=5, label="Significant (FDR<0.05)")
        if np.any(pep_sig):
            # Same label would duplicate legend; only label if we didn't already
            ax.scatter(x[pep_sig], np.asarray(p_pep)[pep_sig],
                      s=14, color="red", edgecolors="none",
                      zorder=5, label=None if np.any(rec_sig) else "Significant (FDR<0.05)")

    ax.set_xlabel(f"Residue position (L={L})")
    ax.set_xlim(1, L)
    ax.set_ylabel("PPI site probability")
    ax.set_ylim(0, 1.05)
    ax.set_title(f"Query: {query_id}\nPartner: {partner_id}")
    ax.grid(True, linestyle=":")
    ax.legend(loc="upper right")
    fig.tight_layout()
    fig.savefig(fig_name)
    plt.close(fig)

    if show_plots:
        from IPython.display import Image, display
        display(Image(filename=fig_name))

def run_ppi_predictions(output_dir,
                        ordered_query_ids=None,
                        ordered_partner_ids=None,
                        num_scrambles=5):
    """
    Loop over all query/partner pairs from the FASTA input widgets
    and write outputs into output_dir.
    """
    # Parse from the global text areas defined in the input cell
    seqs1 = parse_fasta(query_seq_input.value)
    seqs2 = parse_fasta(partner_seq_input.value)

    print(f"Parsed {len(seqs1)} query sequence(s) and {len(seqs2)} partner sequence(s).")

    if not seqs1 or not seqs2:
        print("No sequences found in one of the inputs.")
        update_status("No sequences found in one of the inputs.", busy=False)
        return

    ordered_query_ids   = [sid for _, sid, _ in seqs1]
    ordered_partner_ids = [sid for _, sid, _ in seqs2]

    total_pairs = len(seqs1) * len(seqs2)
    pair_idx = 0

    for (i1, query_id, query_seq), (i2, partner_id, partner_seq) in product(seqs1, seqs2):
        pair_idx += 1
        status_msg = f"Running predictions for {query_id} vs {partner_id} (pair {pair_idx}/{total_pairs})..."
        update_status(status_msg, busy=True)

        print(f"[Query {i1+1}/{len(seqs1)}] [Partner {i2+1}/{len(seqs2)}]")
        run_ppi_for_pair(query_id, query_seq, partner_id, partner_seq,
                        output_dir=output_dir,
                        num_scrambles=num_scrambles)

        done_msg = f"Finished {query_id} vs {partner_id} (pair {pair_idx}/{total_pairs})."
        update_status(done_msg, busy=False)

# Entry point called from the button in the input cell
def run_embedding_workflow(btn=None):
    """
    Callback used by the 'Run predictions' button in the input cell.
    Creates a new timestamped output folder and runs all pairwise predictions.
    """
    global current_output_folder, output_sessions, embedding_cache

    with output_box:
        output_box.clear_output()
        global pred_time_total_s, pred_time_pairs, pred_num_pairs
        pred_time_total_s = 0.0
        pred_time_pairs = {}
        pred_num_pairs = 0

        # New run: clear embedding cache so we don't reuse embeddings
        # from previous runs with different sequences.
        embedding_cache.clear()

        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        current_output_folder = os.path.join("outputs", timestamp)
        os.makedirs(current_output_folder, exist_ok=True)
        output_sessions.append(current_output_folder)

        # Load both models once at the beginning of the workflow
        load_ppi_models()

        print(f"Output folder: {current_output_folder}")
        run_ppi_predictions(output_dir=current_output_folder, num_scrambles=5)

        # ---- Benchmark report ----
        if pred_num_pairs > 0:
            avg = pred_time_total_s / pred_num_pairs
            msg = (
                f"All predictions completed. "
                f"Prediction compute time: {pred_time_total_s:.2f}s total "
                f"({avg:.2f}s per pair; {pred_num_pairs} pair(s))."
            )
        else:
            msg = "All predictions completed."

        update_status(msg, busy=False)



Installing libraries and importing dependencies...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 51.1 MB/s eta 0:00:00
Using device: cpu 

Loading ESM-2 checkpoint: facebook/esm2_t33_650M_UR50D...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/95.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/93.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/724 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.61G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/534 [00:00<?, ?it/s]

[transformers] EsmModel LOAD REPORT from: facebook/esm2_t33_650M_UR50D
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.bias        | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



ESM-2 checkpoint loaded
Note: You may safely ignore the warning about uninitialized pooler weights above. This component is not used in the prediction workflow and does not affect results.

Setup is complete

CPU times: user 32.2 s, sys: 17.1 s, total: 49.3 s
Wall time: 1min 19s


In [4]:
#@title 📝 Input Form
#@markdown > This cell sets the input form and runs the PPI site prediction models.
#@markdown >
#@markdown > *You may provide multiple query and partner sequences. The model will automatically evaluate all possible query–partner combinations.*

# Process Input and Run Predictions

use_null_background = False
show_plots = False  # default; can be toggled via the checkbox below

# Input FASTA Sequences from User
query_seq_input = ipw.Textarea(
    value='>query\nYPCGICTNEVNDDQDAILCEASCQKWFHRICTGMTETAYGLLTAEASAVWGCDTCMA',
    description='Query',
    layout=ipw.Layout(width='80%', height='100px', margin='10px 0px 0px 0px')
)

partner_seq_input = ipw.Textarea(
    value='>partner\nAMAAKVVYVFSTEMANKAAEAVLKGQVETIVSFHI',
    description='Partner(s)',
    layout=ipw.Layout(width='80%', height='100px', margin='10px 0px 0px 0px')
)

use_null_background_checkbox = ipw.Checkbox(
    value=False,
    description='Use null-background distribution for estimating significance (it may be very time consuming for large proteins).',
    indent=False,
    layout=ipw.Layout(margin='10px 0px 0px 0px', width='auto')
)

show_plots_checkbox = ipw.Checkbox(
    value=False,
    description='Show plots for each query–partner pair (inline)',
    indent=False,
    layout=ipw.Layout(margin='5px 0px 0px 0px', width='auto')
)

run_button = ipw.Button(
    description="Run PPI site predictions",
    button_style='primary',
    layout=ipw.Layout(width='200px', margin='5px 0px 20px 0px')
)

status_box = ipw.Output()
output_box = ipw.Output()

display(Markdown("### Input your query and partner protein sequences in FASTA format"))
display(
    query_seq_input,
    partner_seq_input,
    use_null_background_checkbox,
    show_plots_checkbox,
    run_button,
    output_box,
    status_box
)

# Button click handler
def run_embedding_workflow_on_click(b):
    global use_null_background, show_plots
    with output_box:
        output_box.clear_output()
        use_null_background = use_null_background_checkbox.value
        show_plots = show_plots_checkbox.value
        run_embedding_workflow(b)

run_button.on_click(run_embedding_workflow_on_click)



### Input your query and partner protein sequences in FASTA format

Textarea(value='>query\nYPCGICTNEVNDDQDAILCEASCQKWFHRICTGMTETAYGLLTAEASAVWGCDTCMA', description='Query', layou…

Textarea(value='>partner\nAMAAKVVYVFSTEMANKAAEAVLKGQVETIVSFHI', description='Partner(s)', layout=Layout(height…

Checkbox(value=False, description='Use null-background distribution for estimating significance (it may be ver…

Checkbox(value=False, description='Show plots for each query–partner pair (inline)', indent=False, layout=Layo…

Button(button_style='primary', description='Run PPI site predictions', layout=Layout(margin='5px 0px 20px 0px'…

Output()

Output()

In [6]:
#@title 📁 Review and Download Results
#@markdown > This cell retrieves and reviews available PPI site predictions.
#@markdown >
#@markdown > *If you've run multiple PPI predictions, **re-run this cell to refresh the list** of available results in the dropdown menu.*

import zipfile
from glob import glob
from IPython.display import FileLink, display, Markdown, HTML
from google.colab import files

# Widget containers
view_box = ipw.Output()
site_box = ipw.Output()
download_button = ipw.Button(description="⬇️ Download All Files", button_style='success')
download_button.layout.display = 'none'  # Hidden until selection
current_zip_path = {"path": None}
current_txt_files = []

# User-defined filter controls
prob_cutoff = ipw.FloatText(value=0.5, description="Min Probability:", step=0.1, style={'description_width': 'initial'})

fdr_dropdown = ipw.Dropdown(
    options=['None', 0.05, 0.01, 0.001],
    value=0.05 if use_null_background else 'None',
    description='FDR cutoff:',
    style={'description_width': 'initial'},
    layout=ipw.Layout(width='200px', margin='0px 0px 0px 20px')
)

site_button = ipw.Button(description="🔍 Retrieve PPI Sites", button_style='info',
                         layout=ipw.Layout(width='auto', margin='0px 0px 0px 20px'))

# Create ZIP inside outputs/ folder
def create_zip(folder):
    base_name = os.path.basename(folder.rstrip("/"))
    zip_path = f"{folder.rstrip('/')}.zip"
    with zipfile.ZipFile(zip_path, 'w') as zipf:
        for file in glob(f"{folder}/*"):
            arcname = os.path.basename(file)
            zipf.write(file, arcname=arcname)
    return zip_path

# Dropdown callback
def on_select_run(change):
    folder = change['new']
    current_txt_files.clear()
    site_box.clear_output()

    if not folder:
        view_box.clear_output()
        download_button.layout.display = 'none'
        return

    with view_box:
        view_box.clear_output()
        print(f"📂 Files in: {folder}")
        for f in sorted(glob(f"{folder}/*")):
            print(f"• {os.path.basename(f)}")
        print("")

        current_txt_files.extend(sorted(glob(f"{folder}/*.txt")))

        # Create ZIP file
        zip_path = create_zip(folder)
        current_zip_path["path"] = zip_path
        download_button.description = f"Download All ({os.path.basename(zip_path)})"
        download_button.layout.display = 'inline-block'
        download_button.layout = ipw.Layout(width='300px')

# Download button click handler
def on_download_clicked(btn):
    zip_path = current_zip_path.get("path")
    if zip_path and os.path.exists(zip_path):
        files.download(zip_path)

# Find significant residues
def on_site_filter(btn):
    site_box.clear_output()
    cutoff = prob_cutoff.value
    fdr_limit = fdr_dropdown.value
    fdr_active = fdr_limit != 'None'

    with site_box:
        if not current_txt_files:
            print("⚠️ No prediction files loaded.")
            return

        for file_path in current_txt_files:
            results = []
            try:
                with open(file_path) as f:
                    header = f.readline()
                    f.seek(0)
                    has_fdr = "q" in header.lower() or "fdr" in header.lower()

                    for line in f:
                        if line.startswith("#") or line.startswith("Position"):
                            continue
                        parts = line.strip().split('\t')
                        if len(parts) < 3:
                            continue
                        pos, aa, prob = parts[:3]
                        try:
                            prob = float(prob)
                            qval = float(parts[4]) if has_fdr and len(parts) > 4 else None
                            if prob >= cutoff and (not fdr_active or qval is None or qval < fdr_limit):
                                results.append((pos, aa, prob, qval if has_fdr else None))
                        except:
                            continue

                if results:
                    print(f"📄 {os.path.basename(file_path)} - {len(results)} site(s) found:")
                    header = "  Res\tProb" + ("\tFDR" if results[0][3] is not None else "")
                    print(header)
                    for r in results:
                        row = f"  {r[1]}{r[0]}\t{r[2]:.3f}"
                        if r[3] is not None:
                            row += f"\t{r[3]:.3g}"
                        print(row)
                    print("")
            except Exception as e:
                print(f"⚠️ Error reading {file_path}: {e}")


# Hook up actions
download_button.on_click(on_download_clicked)
site_button.on_click(on_site_filter)

# List all subfolders under outputs/
output_sessions = sorted(glob("outputs/*/"))
session_selector = ipw.Dropdown(
    options=["Choose submission from the list"] + output_sessions,
    description=f'Select submission ({len(output_sessions)} found):',
    layout=ipw.Layout(min_width='400px', max_width='800px'),
    style={'description_width': 'initial'}
)
session_selector.observe(on_select_run, names='value')

# Display UI
display(ipw.VBox([
    session_selector,
    view_box,
    ipw.HBox([prob_cutoff, fdr_dropdown, site_button]),
    site_box,
    download_button
]))


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>